# Exercise 11

## Group ID: 22
*   Taha El Amine Kassabi
*   Mohamed Hazem Badawi
*   Carolin Goj

## Exercise day: Tuesday

## Description
In this exercise you will implement a Tokenizer class that will be used to tokenize a given text.
A token will be a short sequence of bytes and tokenization will be about splitting up the text into those short substrings and then representing it in terms of token ids.
This allows language models to work on a better text representation than just using a character level model as in the last exercise.
We will use the so called "Byte Pair Encoding" (BPE) algorithm to tokenize the text. The BPE algorithm is a simple algorithm that iteratively merges the most frequent pair of bytes/ids in the text. This notebook will guide you through the implementation of the Tokenizer class. To make the representation of the text easier, we will use integer ids to represent the bytes of the text.

Note: The tokenizer is not part of the model, it is a preprocessing step that is used to tokenize the text before feeding it to the model. Therefore, it can be trained on a different dataset than the model.
Once trained, the tokenizer is capable of encoding and decoding any given text to/from a list of tokens.

## Tasks
1. Implement the `get_stats` method that will return a dictionary containing the frequency of each pair of characters in the text. (0.5 points)
1. Implement the `merge` method that will merge the most frequent pair of characters in the text. (0.5 points)
1. Implement the `fit` method that will train the tokenizer on the given text. (1 point)
1. Implement the `encode` method that will encode the given text to a list of tokens. (1 point)
1. Implement the `decode` method that will decode the given list of tokens to a text. (1 point)
1. Implement the `BPE` class. This class should contain the previously implemented methods. (1 point)

In [1]:
from collections import Counter
import functools
import itertools

from tqdm.auto import tqdm

In [2]:
def get_stats(ids):
    """ Returns a dictionary with the number of times each pair of ids appears in the input list """
    return Counter(zip(ids, ids[1:]))

In [3]:
ids = [1, 2, 3, 1, 2, 3, 1, 2, 3]
expected = {(1, 2): 3, (2, 3): 3, (3, 1): 2}
assert get_stats(ids) == expected

In [4]:
def merge(ids, pair, new_id):
    """Merge every pair of elements in ids that are equal to pair into a single element new_id"""
    # too slow
    # return functools.reduce(lambda acc, x: acc[:-1] + [new_id] if tuple(acc[-1:] + [x]) == pair else acc + [x], ids, [])

    acc, new_id = [], (new_id,)

    for id in ids:
        acc += [id]
        if tuple(acc[-2:]) == pair:
            acc[-2:] = new_id

    return acc

In [5]:
ids = [0, 1, 2, 3, 0, 1, 2, 3]
pair = (0, 1)
new_id = 4
new_ids = merge(ids, pair, new_id)
expected = [4, 2, 3, 4, 2, 3]
assert new_ids == expected, f"Merge failed new_ids actual: {new_ids} expected: {expected}"

In [6]:
def fit(ids, max_iter=1000):
    """Fit the model to the data by merging recursively the most common pairs of ids max_iter times or until no pair appears more than once. If two pairs have the same frequency, the one that appears first is chosen.
    Returns the fitted model and a dictionary with the merges containing the new_id for each pair.
    To ensure each new_id is unique (and our results are comparable), it is set to the maximum id in the list plus one.
    """
    start_id, merges = max(ids) + 1, {}
    for next_id in tqdm(range(start_id, start_id + max_iter), desc='Fitting model'):
        stats = get_stats(ids)
        max_pair = max(stats, key=lambda v: stats[v])
        ids = merge(ids, max_pair, next_id)
        merges[max_pair] = next_id
        if len(ids) == 1: break
    return ids, merges

In [7]:
ids = [0, 1, 2, 3, 0, 1, 2, 3]
num_iterations = 1
new_ids, merges = fit(ids, num_iterations)
assert new_ids == [4, 2, 3, 4, 2, 3], f"Wrong ids after fit {new_ids}"
assert merges == {(0, 1): 4}, f"Wrong merges after fit {merges}"

Fitting model:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
ids = [0, 1, 2, 3, 0, 1, 2, 3]
num_iterations = 2
new_ids, merges = fit(ids, num_iterations)
assert new_ids == [5, 3, 5, 3], f"Wrong ids after fit {new_ids}"
assert merges == {(0, 1): 4, (4, 2): 5}, f"Wrong merges after fit {merges}"

Fitting model:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
def encode(ids, merges):
    """Encode the input list of ids using the merges dictionary"""

    def replace_n_1_recursive(acc, x):
        if tuple(acc[-2:]) in merges:
            acc = replace_n_1_recursive(acc[:-2] + [merges[tuple(acc[-2:])]], x)
        elif tuple(acc[-1:] + [x]) in merges:
            acc = replace_n_1_recursive(acc[:-1], merges[tuple(acc[-1:] + [x])])
        else:
            acc += [x]
        return acc

    return functools.reduce(replace_n_1_recursive, ids, [])

In [10]:
ids = [0, 1, 2, 3, 0, 1, 2, 3]
merges = {(0, 1): 4}
encoded_ids = encode(ids, merges)
expected = [4, 2, 3, 4, 2, 3]
assert encoded_ids == expected, f"Wrong encoded ids {encoded_ids} expected: {expected}"

In [11]:
# I added these tests because recursive replacements are a pivotal check
ids = [0, 1, 2, 3, 0, 1, 2, 3]
merges = {(0, 1): 4, (4, 2): 5}
encoded_ids = encode(ids, merges)
expected = [5, 3, 5, 3]
assert encoded_ids == expected, f"Wrong encoded ids {encoded_ids} expected: {expected}"

In [12]:
ids = [0, 1, 2, 3, 0, 1, 2, 3]
merges = {(0, 1): 4, (2, 3): 5, (4, 5): 6}
encoded_ids = encode(ids, merges)
expected = [6, 6]
assert encoded_ids == expected, f"Wrong encoded ids {encoded_ids} expected: {expected}"

In [13]:
def decode(ids, merges):
    """Decode the input list of ids using the merges dictionary"""
    merges = {v: k for k, v in merges.items()}

    def replace_1_n_recursive(id):
        return itertools.chain(*map(replace_1_n_recursive, merges[id])) if id in merges else [id]

    return list(itertools.chain(*map(replace_1_n_recursive, ids)))

In [14]:
ids = [6, 6]
merges = {(0, 1): 4, (4, 2): 5, (5, 3): 6}
decoded_ids = decode(ids, merges)
expected = [0, 1, 2, 3, 0, 1, 2, 3]
assert decoded_ids == expected, f"Wrong decoded ids {decoded_ids} expected: {expected}"

In [15]:
class BPE:
    def __init__(self, max_iter: int = 1000):
        self.max_iter = max_iter
        self.vocab = {}

    def fit(self, text):
        text = list(map(ord, text))
        vals, self.vocab = fit(text, max_iter=self.max_iter)
        return vals

    def encode(self, text):
        text = list(map(ord, text))
        text = tqdm(text, desc='Encode')
        return encode(text, self.vocab)

    def decode(self, ids):
        ids = tqdm(ids, desc='Decode')
        ids = decode(ids, self.vocab)
        return ''.join(map(chr, ids))

In [16]:
text = "A large language model (LLM) is a type of computational model designed for natural language processing tasks such as language generation. As language models, LLMs acquire these abilities by learning statistical relationships from vast amounts of text during a self-supervised and semi-supervised training process. The largest and most capable LLMs are artificial neural networks built with a decoder-only transformer-based architecture, enabling efficient processing and generation of large-scale text data. Modern models can be fine-tuned for specific tasks or guided by prompt engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data they are trained in."  # cleaned version of the first paragraph of the Wikipedia page on LLMs
ids = list(map(int, text.encode('utf-8')))

In [17]:
bpe = BPE(max_iter=1000)
bpe.fit(text)
encoded_text = bpe.encode(text)
decoded_text = bpe.decode(encoded_text)
assert text == decoded_text, "Decoding failed"

Fitting model:   0%|          | 0/1000 [00:00<?, ?it/s]

Encode:   0%|          | 0/796 [00:00<?, ?it/s]

Decode:   0%|          | 0/345 [00:00<?, ?it/s]

## Evaluation: 

Let us check what a tokenizer will give us for training a simple language model.

We will use the tiny Shakespeare dataset and train with and without tokenization and compare generated text.


In [18]:
!wget 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

--2025-01-08 10:13:04--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.7’

input.txt.7         100%[===================>]   1.06M  6.66MB/s    in 0.2s    

2025-01-08 10:13:05 (6.66 MB/s) - ‘input.txt.7’ saved [1115394/1115394]



In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

mps


In [20]:
# Compute tokenizer
with open('input.txt') as f:
    text = f.read()
bpe = BPE(max_iter=1000)
bpe.fit(text)
enc = bpe.encode(text)

Fitting model:   0%|          | 0/1000 [00:00<?, ?it/s]

Encode:   0%|          | 0/1115394 [00:00<?, ?it/s]

In [21]:
print(enc[:1000])

[70, 166, 163, 32, 1045, 134, 851, 522, 405, 452, 111, 231, 291, 160, 110, 132, 1066, 114, 230, 114, 128, 488, 114, 570, 654, 441, 198, 820, 142, 836, 854, 665, 128, 721, 198, 10, 70, 166, 163, 32, 1045, 134, 901, 787, 553, 32, 176, 518, 108, 118, 291, 32, 197, 230, 114, 917, 219, 105, 123, 640, 917, 280, 334, 426, 348, 820, 142, 134, 82, 147, 270, 118, 291, 237, 176, 518, 108, 118, 291, 198, 10, 70, 166, 163, 32, 1045, 134, 70, 166, 163, 128, 401, 107, 165, 119, 32, 67, 97, 105, 308, 32, 653, 398, 308, 32, 187, 226, 104, 105, 627, 32, 137, 283, 132, 396, 258, 265, 854, 382, 188, 198, 820, 142, 134, 718, 107, 165, 119, 403, 128, 405, 107, 165, 119, 403, 198, 10, 70, 166, 163, 32, 1045, 134, 76, 320, 32, 308, 32, 107, 250, 108, 342, 128, 284, 155, 707, 175, 965, 123, 99, 734, 840, 484, 298, 234, 202, 231, 198, 73, 115, 403, 583, 263, 271, 371, 348, 820, 142, 729, 136, 394, 176, 258, 1046, 240, 141, 403, 242, 985, 32, 183, 487, 219, 708, 408, 588, 304, 128, 588, 304, 324, 10, 83, 680, 14

In [22]:
print(bpe.decode(enc[:200]))

Decode:   0%|          | 0/200 [00:00<?, ?it/s]

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away


In [23]:
class TextDataset(Dataset):
    def __init__(self, enc_text, blocksize):
        self.data = enc_text
        self.blocksize = blocksize

    def __len__(self):
        return len(self.data) - self.blocksize - 1

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.blocksize]
        y = self.data[idx + self.blocksize]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long),

In [24]:
class MLPNextTokenPredictor(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim=32, hidden_dim=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim

        self.emb = nn.Embedding(vocab_size, embed_dim)

        self.fc1 = nn.Linear(block_size * embed_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.emb(x)
        emb_flat = emb.view(-1, self.block_size * self.embed_dim)

        h = torch.relu(self.bn1(self.fc1(emb_flat)))
        h = torch.relu(self.bn2(self.fc2(h)))
        logits = self.fc3(h)

        return logits

In [25]:
# generate new text based on continuing provided one
def generate(model, block_size, starting_text):
    model.eval()
    assert len(starting_text) >= block_size
    x = torch.tensor(starting_text, dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(100):
            logits = model(x[-block_size:])
            next_token = torch.multinomial(F.softmax(logits, dim=-1), 1)
            starting_text = starting_text + [next_token.item()]
            x = torch.cat([x, next_token.squeeze(0)])
    return starting_text

In [26]:
def produce_example_text(model, block_size, tokenizer):
    starting_text = """We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance;'
"""
    tokenized_starting_text = tokenizer.encode(starting_text)
    continuation = generate(model, block_size, tokenized_starting_text)
    return tokenizer.decode(continuation[len(tokenized_starting_text):])

In [27]:
# Compare with untrained MLP:
vocab_size = len(bpe.vocab)
block_size = 32
mlp_untrained = MLPNextTokenPredictor(vocab_size, block_size)
mlp_untrained.to(device)
continuation = produce_example_text(mlp_untrained, block_size, bpe)
print(continuation)

Encode:   0%|          | 0/352 [00:00<?, ?it/s]

Decode:   0%|          | 0/100 [00:00<?, ?it/s]

ectreatellhdiindhere &eaatlet queenVOLorkOrumThanwhI As ous as INIUS:
He 

TIO:
doe, and ruwill tlord, ADendCome, ROME theirnoteed they would  your fulverS:
And toof the We otherafA:
By art rowne?

death forwith ightttsclifightithit is ow verest uttis the4es
First more tunmuchorETeakasothboso ouurn`poET:
thy ARD Iay, ES:
Govery d,  we 


In [28]:
def train(model, dataloader, optimizer, nr_epochs):
    model.train()
    for epoch in tqdm(range(nr_epochs), desc='Epoch'):
        avg_loss = 0.0
        for x, y in tqdm(dataloader, desc='Batch', leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, model.vocab_size), y.view(-1))
            loss.backward()
            optimizer.step()
            avg_loss += loss.item() / len(dataloader)
        print(f"Epoch {epoch} loss: {avg_loss}")

In [29]:
# training on tokenized text
block_size = 32
vocab_size = len(bpe.vocab)
nr_epochs = 15

dataloader = DataLoader(TextDataset(enc, block_size), batch_size=32, shuffle=True, drop_last=True)
mlp_tokenizer = MLPNextTokenPredictor(vocab_size, block_size)
mlp_tokenizer.to(device)
optimizer = optim.AdamW(mlp_tokenizer.parameters(), lr=0.001, weight_decay=0.01)

train(mlp_tokenizer, dataloader, optimizer, nr_epochs)

Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Batch:   0%|          | 0/16459 [00:00<?, ?it/s]

Epoch 0 loss: 4.1987265079509415


Batch:   0%|          | 0/16459 [00:00<?, ?it/s]

Epoch 1 loss: 3.5464031537991767


...

Epoch 12 loss: 3.1074481260675864


Batch:   0%|          | 0/16459 [00:00<?, ?it/s]

Epoch 13 loss: 3.1006820525844296


Batch:   0%|          | 0/16459 [00:00<?, ?it/s]

Epoch 14 loss: 3.096147609101764


In [30]:
# generate new text using tokenization based MLP
continuation = produce_example_text(mlp_tokenizer, block_size, bpe)
print(continuation)

Encode:   0%|          | 0/352 [00:00<?, ?it/s]

Decode:   0%|          | 0/100 [00:00<?, ?it/s]

excuser's deimestiones, good damdmeh, that knows
Than vexreta my brothers hols, to greath's now
And art her manneed. Well heart of all tongubdie,
'll bedeemued signiory! man!

PbLIither need there.




In [31]:
# training on original text
block_size = 64  # let us give more context because of no tokenization
# convert text to numbers
char_enc = [ord(c) for c in text]
vocab_size = max(char_enc) + 1
nr_epochs = 15

dataloader = DataLoader(TextDataset(char_enc, block_size), batch_size=32, shuffle=True, drop_last=True)
mlp_wo_tokenizer = MLPNextTokenPredictor(vocab_size, block_size)
mlp_wo_tokenizer.to(device)
optimizer = optim.AdamW(mlp_wo_tokenizer.parameters(), lr=0.001, weight_decay=0.01)

train(mlp_wo_tokenizer, dataloader, optimizer, nr_epochs)

Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Batch:   0%|          | 0/34854 [00:00<?, ?it/s]

Epoch 0 loss: 2.058869984663371


Batch:   0%|          | 0/34854 [00:00<?, ?it/s]

Epoch 1 loss: 1.8176167923868487


...

Epoch 12 loss: 1.6797572691041713


Batch:   0%|          | 0/34854 [00:00<?, ?it/s]

Epoch 13 loss: 1.6785148054715502


Batch:   0%|          | 0/34854 [00:00<?, ?it/s]

Epoch 14 loss: 1.677048652303905


In [32]:
# generate new text using non-tokenization based MLP
class no_tok:
    def encode(self, text):
        return [ord(c) for c in text]

    def decode(self, ids):
        return ''.join([chr(i) for i in ids])


t = produce_example_text(mlp_wo_tokenizer, block_size, no_tok())
print(t)

I worthy a pooes if hith the sprozy it me,
That how, curty Rath shuse and unto woo.

SEBASTIAN:
Pons
